In [ ]:
# ============================================================
# HÜCRE 1: Kütüphaneler & Ham Veri Yükleme
# ============================================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# --- Veri Yolu ---
DATA_PATH = r"ALL_TICKERS_2010_2026.csv"
OUTPUT_PATH = r"derin_ogrenme_hazir_veri.csv"

# --- Ham Veriyi Oku ---
raw = pd.read_csv(DATA_PATH, parse_dates=["Date"])
print(f"Ham veri boyutu  : {raw.shape}")
print(f"Sütunlar         : {list(raw.columns)}")
print(f"Ticker sayısı    : {raw['Ticker'].nunique()} → {sorted(raw['Ticker'].unique())}")
print(f"Tarih aralığı    : {raw['Date'].min().date()} → {raw['Date'].max().date()}")
raw.head()

Ham veri boyutu  : (47653, 7)
Sütunlar         : ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker']
Ticker sayısı    : 12 → ['CL=F', 'GLD', 'ITA', 'SPY', 'XLE', 'XLF', 'XLK', 'XLP', 'XLRE', 'XLV', 'XLY', '^VIX']
Tarih aralığı    : 2010-01-04 → 2026-04-10


,Date,Close,High,Low,Open,Volume,Ticker
0,2010-01-04,84.796349,84.841240,83.434579,84.078053,118944600,SPY
1,2010-01-05,85.020828,85.058242,84.437213,84.743989,111579900,SPY
2,2010-01-06,85.080727,85.290229,84.871224,84.938562,116074400,SPY
3,2010-01-07,85.439850,85.544601,84.684141,84.923573,131091100,SPY
4,2010-01-08,85.724167,85.761580,85.043285,85.215373,126402800,SPY


In [2]:
# ============================================================
# HÜCRE 2: Veri Şekillendirme — Drop & Pivot
# ============================================================

# 1) High, Low, Open sütunlarını çöpe at
cols_to_drop = [c for c in ["High", "Low", "Open", "Adj Close"] if c in raw.columns]
df_slim = raw.drop(columns=cols_to_drop)
print(f"Kalan sütunlar: {list(df_slim.columns)}")

# 2) Close ve Volume için ayrı pivot tablolar oluştur
pivot_close = df_slim.pivot_table(
    index="Date", columns="Ticker", values="Close", aggfunc="first"
)
pivot_volume = df_slim.pivot_table(
    index="Date", columns="Ticker", values="Volume", aggfunc="first"
)

# 3) Sütun isimlerini "TICKER_Close", "TICKER_Volume" formatına çevir
pivot_close.columns = [f"{t}_Close" for t in pivot_close.columns]
pivot_volume.columns = [f"{t}_Volume" for t in pivot_volume.columns]

# 4) Yan yana birleştir
df_wide = pd.concat([pivot_close, pivot_volume], axis=1)

# 5) Sütunları mantıksal sıraya koy: her ticker için Close, Volume yan yana
ticker_order = ["SPY", "^VIX", "XLK", "XLV", "ITA", "XLE", "XLF", "XLP", "XLY", "XLRE", "GLD", "CL=F"]
ordered_cols = []
for t in ticker_order:
    ordered_cols.append(f"{t}_Close")
    ordered_cols.append(f"{t}_Volume")
df_wide = df_wide[ordered_cols]

# 6) XLRE Kasım 2015 kesimi
df_wide = df_wide.loc["2015-11-01":]

print(f"Pivot sonrası boyut : {df_wide.shape}")
print(f"Tarih aralığı       : {df_wide.index.min().date()} → {df_wide.index.max().date()}")
print(f"NaN toplam          : {df_wide.isnull().sum().sum()}")
df_wide.head()

Kalan sütunlar: ['Date', 'Close', 'Volume', 'Ticker']
Pivot sonrası boyut : (2626, 24)
Tarih aralığı       : 2015-11-02 → 2026-04-10
NaN toplam          : 48


,SPY_Close,SPY_Volume,^VIX_Close,^VIX_Volume,XLK_Close,XLK_Volume,XLV_Close,XLV_Volume,ITA_Close,ITA_Volume,...,XLP_Close,XLP_Volume,XLY_Close,XLY_Volume,XLRE_Close,XLRE_Volume,GLD_Close,GLD_Volume,CL=F_Close,CL=F_Volume
Date,,,,,,,,,,,,,,,,,,,,,
2015-11-02,176.775894,86270800.0,14.15,0.0,19.552010,27486600.0,61.174213,18646600.0,53.790512,143800.0,...,37.926029,13054000.0,36.296738,15470000.0,21.559605,100.0,108.589996,5843300.0,46.139999,305518.0
2015-11-03,177.288422,95246100.0,14.54,0.0,19.654112,13758000.0,60.955898,10428700.0,53.651146,250400.0,...,37.744637,12452900.0,36.435074,14567000.0,21.838243,500.0,106.980003,7145300.0,47.900002,434574.0
2015-11-04,176.750671,96224500.0,15.51,0.0,19.676311,14597200.0,60.687225,11533000.0,53.529778,119800.0,...,37.585915,8067600.0,36.198566,12531800.0,21.747681,1900.0,105.970001,7617800.0,46.320000,459307.0
2015-11-05,176.574188,78408700.0,15.05,0.0,19.618595,13457600.0,60.494133,9172000.0,53.718582,102600.0,...,37.578365,9691800.0,36.327984,13929400.0,21.664099,23800.0,105.639999,5931100.0,45.200001,437646.0
2015-11-06,176.481812,110471500.0,14.33,0.0,19.685186,13737800.0,60.242241,12661100.0,53.601700,94200.0,...,37.170231,13747200.0,36.332432,13186400.0,20.863012,9000.0,104.099998,8831900.0,44.290001,448133.0


In [3]:
# ============================================================
# HÜCRE 3: Yardımcı Fonksiyonlar
# ============================================================

def log_return(series: pd.Series) -> pd.Series:
    """Logaritmik getiri: ln(Pt / Pt-1)"""
    return np.log(series / series.shift(1))


def abnormal_return(asset_log_ret: pd.Series, benchmark_log_ret: pd.Series) -> pd.Series:
    """Anormal Getiri = Varlık Log Getirisi - SPY Log Getirisi"""
    return asset_log_ret - benchmark_log_ret


def rolling_volatility(log_ret: pd.Series, window: int) -> pd.Series:
    """Yuvarlanan pencere volatilitesi (standart sapma)"""
    return log_ret.rolling(window=window).std()


def rsi_14(close: pd.Series, period: int = 14) -> pd.Series:
    """
    Klasik RSI hesaplaması (Wilder's smoothing).
    RSI = 100 - (100 / (1 + RS))
    RS  = Ortalama Kazanç / Ortalama Kayıp
    """
    delta = close.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)

    # İlk 'period' günün basit ortalamasıyla başla, sonra EWM (Wilder)
    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()

    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))


def sma_distance(close: pd.Series, window: int = 20) -> pd.Series:
    """SMA'dan yüzdesel sapma: (P - SMA) / SMA"""
    sma = close.rolling(window=window).mean()
    return (close - sma) / sma


def safe_log_volume(volume: pd.Series) -> pd.Series:
    """
    Güvenli logaritmik hacim.
    Volume=0 olan yerlerde (VIX gibi) ln(0)=-inf üretir.
    Bu değerleri NaN yapıp ffill ile dolduruyoruz.
    """
    log_vol = np.log(volume)
    log_vol.replace([np.inf, -np.inf], np.nan, inplace=True)
    log_vol = log_vol.ffill()
    # Eğer ilk satırlarda da NaN varsa 0 ile doldur
    log_vol = log_vol.fillna(0)
    return log_vol


print("✅ Tüm yardımcı fonksiyonlar tanımlandı.")

✅ Tüm yardımcı fonksiyonlar tanımlandı.


In [4]:
# ============================================================
# HÜCRE 4: Teknik Özellik Çıkarımı (Feature Engineering)
# ============================================================

# SPY ve VIX HARİÇ, özellik hesaplanacak ticker'lar
feature_tickers = ["XLK", "XLV", "ITA", "XLE", "XLF", "XLP", "XLY", "XLRE", "GLD", "CL=F"]

# Önce SPY log getirisini hesapla (benchmark)
df_wide["SPY_Log_Getiri"] = log_return(df_wide["SPY_Close"])

for t in feature_tickers:
    close_col = f"{t}_Close"
    vol_col = f"{t}_Volume"
    prefix = t.replace("^", "").replace("=", "_")  # sütun ismi güvenliği

    # 1) Logaritmik Getiri
    df_wide[f"{prefix}_Log_Getiri"] = log_return(df_wide[close_col])

    # 2) Anormal Getiri (AR)
    df_wide[f"{prefix}_AR"] = abnormal_return(
        df_wide[f"{prefix}_Log_Getiri"],
        df_wide["SPY_Log_Getiri"]
    )

    # 3) Volatilite 10 gün
    df_wide[f"{prefix}_Volatilite_10g"] = rolling_volatility(
        df_wide[f"{prefix}_Log_Getiri"], window=10
    )

    # 4) Volatilite 30 gün
    df_wide[f"{prefix}_Volatilite_30g"] = rolling_volatility(
        df_wide[f"{prefix}_Log_Getiri"], window=30
    )

    # 5) RSI 14
    df_wide[f"{prefix}_RSI_14"] = rsi_14(df_wide[close_col])

    # 6) SMA Uzaklık 20
    df_wide[f"{prefix}_SMA_Uzaklik_20"] = sma_distance(df_wide[close_col], window=20)

    # 7) Log Hacim
    df_wide[f"{prefix}_Log_Hacim"] = safe_log_volume(df_wide[vol_col])

    print(f"✅ {t}: 7 özellik eklendi")

print(f"\nToplam sütun sayısı: {len(df_wide.columns)}")
print(f"DataFrame boyutu  : {df_wide.shape}")

✅ XLK: 7 özellik eklendi
✅ XLV: 7 özellik eklendi
✅ ITA: 7 özellik eklendi
✅ XLE: 7 özellik eklendi
✅ XLF: 7 özellik eklendi
✅ XLP: 7 özellik eklendi
✅ XLY: 7 özellik eklendi
✅ XLRE: 7 özellik eklendi
✅ GLD: 7 özellik eklendi
✅ CL=F: 7 özellik eklendi

Toplam sütun sayısı: 95
DataFrame boyutu  : (2626, 95)


In [5]:
# ============================================================
# HÜCRE 5: Sütun Envanteri
# ============================================================
feature_cols = [c for c in df_wide.columns if any(
    tag in c for tag in ["Log_Getiri", "AR", "Volatilite", "RSI", "SMA_Uzaklik", "Log_Hacim"]
)]
print(f"Toplam özellik sütunu: {len(feature_cols)}\n")
for i, c in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {c}")

Toplam özellik sütunu: 71

   1. SPY_Log_Getiri
   2. XLK_Log_Getiri
   3. XLK_AR
   4. XLK_Volatilite_10g
   5. XLK_Volatilite_30g
   6. XLK_RSI_14
   7. XLK_SMA_Uzaklik_20
   8. XLK_Log_Hacim
   9. XLV_Log_Getiri
  10. XLV_AR
  11. XLV_Volatilite_10g
  12. XLV_Volatilite_30g
  13. XLV_RSI_14
  14. XLV_SMA_Uzaklik_20
  15. XLV_Log_Hacim
  16. ITA_Log_Getiri
  17. ITA_AR
  18. ITA_Volatilite_10g
  19. ITA_Volatilite_30g
  20. ITA_RSI_14
  21. ITA_SMA_Uzaklik_20
  22. ITA_Log_Hacim
  23. XLE_Log_Getiri
  24. XLE_AR
  25. XLE_Volatilite_10g
  26. XLE_Volatilite_30g
  27. XLE_RSI_14
  28. XLE_SMA_Uzaklik_20
  29. XLE_Log_Hacim
  30. XLF_Log_Getiri
  31. XLF_AR
  32. XLF_Volatilite_10g
  33. XLF_Volatilite_30g
  34. XLF_RSI_14
  35. XLF_SMA_Uzaklik_20
  36. XLF_Log_Hacim
  37. XLP_Log_Getiri
  38. XLP_AR
  39. XLP_Volatilite_10g
  40. XLP_Volatilite_30g
  41. XLP_RSI_14
  42. XLP_SMA_Uzaklik_20
  43. XLP_Log_Hacim
  44. XLY_Log_Getiri
  45. XLY_AR
  46. XLY_Volatilite_10g
  47. XLY_Volatil

In [6]:
# ============================================================
# HÜCRE 6: Olay Çalışması (Event Study) — Tanımlar
# ============================================================

olaylar = {
    '2016-06-23': 'Brexit_Referandumu_2016',
    '2016-11-08': 'ABD_Secimleri_2016',
    '2018-02-05': 'Volmageddon_VIX_Patlamasi_2018',
    '2018-03-22': 'ABD_Cin_Ticaret_Savasi_2018',
    '2019-09-16': 'Aramco_Petrol_Soku_2019',
    '2020-03-11': 'COVID19_Pandemisi_2020',
    '2022-02-24': 'Rusya_Ukrayna_Savasi_2022',
    '2023-03-10': 'SVB_Bankacilik_Krizi_2023',
    '2023-05-25': 'Yapay_Zeka_Soku_Nvidia_2023',
    '2023-08-01': 'ABD_Kredi_Notu_Dususu_2023',
    '2023-10-09': 'Israil_Hamas_Savasi_2023',
    '2024-04-15': 'Israil_Iran_2024',
    '2024-08-05': 'Kuresel_Flash_Crash_2024',
    '2024-09-18': 'Fed_Faiz_Pivotu_2024',
    '2024-11-05': 'ABD_Secimleri_2024',
}

def find_t0(target_date_str: str, date_index: pd.DatetimeIndex) -> pd.Timestamp:
    """
    Tatil Günü Mantığı:
    Verilen tarih indekste yoksa (hafta sonu / tatil),
    bir sonraki ilk işlem gününü T0 olarak döndürür.
    """
    target = pd.Timestamp(target_date_str)
    if target in date_index:
        return target
    # İndekste target'ten büyük olan ilk tarihi bul
    future_dates = date_index[date_index > target]
    if len(future_dates) == 0:
        raise ValueError(f"{target_date_str} sonrasında işlem günü bulunamadı!")
    return future_dates[0]

# T0 tarihlerini hesapla ve göster
print(f"{'Olay Tarihi':<14} {'T0 (İşlem Günü)':<18} {'Olay Adı'}")
print("=" * 70)
for date_str, event_name in olaylar.items():
    t0 = find_t0(date_str, df_wide.index)
    shift = (t0 - pd.Timestamp(date_str)).days
    flag = f" ⟶ +{shift} gün kaydı" if shift > 0 else ""
    print(f"{date_str:<14} {str(t0.date()):<18} {event_name}{flag}")

Olay Tarihi    T0 (İşlem Günü)    Olay Adı
2016-06-23     2016-06-23         Brexit_Referandumu_2016
2016-11-08     2016-11-08         ABD_Secimleri_2016
2018-02-05     2018-02-05         Volmageddon_VIX_Patlamasi_2018
2018-03-22     2018-03-22         ABD_Cin_Ticaret_Savasi_2018
2019-09-16     2019-09-16         Aramco_Petrol_Soku_2019
2020-03-11     2020-03-11         COVID19_Pandemisi_2020
2022-02-24     2022-02-24         Rusya_Ukrayna_Savasi_2022
2023-03-10     2023-03-10         SVB_Bankacilik_Krizi_2023
2023-05-25     2023-05-25         Yapay_Zeka_Soku_Nvidia_2023
2023-08-01     2023-08-01         ABD_Kredi_Notu_Dususu_2023
2023-10-09     2023-10-09         Israil_Hamas_Savasi_2023
2024-04-15     2024-04-15         Israil_Iran_2024
2024-08-05     2024-08-05         Kuresel_Flash_Crash_2024
2024-09-18     2024-09-18         Fed_Faiz_Pivotu_2024
2024-11-05     2024-11-05         ABD_Secimleri_2024


In [7]:
# ============================================================
# HÜCRE 7: 31 Günlük Olay Penceresi Kesimi
# ============================================================

event_windows = []

for date_str, event_name in olaylar.items():
    t0 = find_t0(date_str, df_wide.index)
    t0_loc = df_wide.index.get_loc(t0)

    # T-15 ve T+15 sınırlarını kontrol et
    start_loc = t0_loc - 15
    end_loc = t0_loc + 15

    if start_loc < 0:
        print(f"⚠️  {event_name}: T-15 verinin dışında, atlanıyor!")
        continue
    if end_loc >= len(df_wide):
        print(f"⚠️  {event_name}: T+15 verinin dışında, atlanıyor!")
        continue

    # 31 günlük dilimi kes
    window = df_wide.iloc[start_loc : end_loc + 1].copy()

    # Meta sütunları ekle
    window["Olay_Ismi"] = event_name
    window["T0_Goreceli_Gun"] = list(range(-15, 16))  # -15, -14, ..., 0, ..., +14, +15
    window["T0_Tarih"] = t0

    event_windows.append(window)
    print(f"✅ {event_name:<40} | T0={t0.date()} | {len(window)} satır kesildi")

print(f"\nToplam olay penceresi: {len(event_windows)}")
print(f"Beklenen toplam satır: {len(event_windows)} × 31 = {len(event_windows) * 31}")

✅ Brexit_Referandumu_2016                  | T0=2016-06-23 | 31 satır kesildi
✅ ABD_Secimleri_2016                       | T0=2016-11-08 | 31 satır kesildi
✅ Volmageddon_VIX_Patlamasi_2018           | T0=2018-02-05 | 31 satır kesildi
✅ ABD_Cin_Ticaret_Savasi_2018              | T0=2018-03-22 | 31 satır kesildi
✅ Aramco_Petrol_Soku_2019                  | T0=2019-09-16 | 31 satır kesildi
✅ COVID19_Pandemisi_2020                   | T0=2020-03-11 | 31 satır kesildi
✅ Rusya_Ukrayna_Savasi_2022                | T0=2022-02-24 | 31 satır kesildi
✅ SVB_Bankacilik_Krizi_2023                | T0=2023-03-10 | 31 satır kesildi
✅ Yapay_Zeka_Soku_Nvidia_2023              | T0=2023-05-25 | 31 satır kesildi
✅ ABD_Kredi_Notu_Dususu_2023               | T0=2023-08-01 | 31 satır kesildi
✅ Israil_Hamas_Savasi_2023                 | T0=2023-10-09 | 31 satır kesildi
✅ Israil_Iran_2024                         | T0=2024-04-15 | 31 satır kesildi
✅ Kuresel_Flash_Crash_2024                 | T0=2024-08-05 | 31 

In [ ]:
# ============================================================
# HÜCRE 8: Final — Concat, Temizlik, CSV Export
# ============================================================

# 1) Tüm olay pencerelerini alt alta birleştir
df_final = pd.concat(event_windows, ignore_index=False)

# Date'i sütun olarak da tut (indeksten)
df_final.insert(0, "Date", df_final.index)
df_final = df_final.reset_index(drop=True)

print(f"Birleştirme sonrası boyut : {df_final.shape}")
print(f"NaN sayısı (temizlik öncesi) : {df_final.isnull().sum().sum()}")
print(f"Inf sayısı (temizlik öncesi) : {np.isinf(df_final.select_dtypes(include=[np.number])).sum().sum()}")

# 2) Inf → NaN dönüşümü
df_final.replace([np.inf, -np.inf], np.nan, inplace=True)

# 3) NaN temizliği — satır bazlı silme
rows_before = len(df_final)
df_final.dropna(inplace=True)
rows_after = len(df_final)
print(f"Silinen satır sayısı         : {rows_before - rows_after}")

# 4) Son kontrol
assert df_final.isnull().sum().sum() == 0, "HATA: Hâlâ NaN var!"
assert np.isinf(df_final.select_dtypes(include=[np.number])).sum().sum() == 0, "HATA: Hâlâ inf var!"

print(f"\n{'='*50}")
print(f"✅ Final DataFrame boyutu    : {df_final.shape}")
print(f"✅ Toplam olay sayısı        : {df_final['Olay_Ismi'].nunique()}")
print(f"✅ NaN                       : {df_final.isnull().sum().sum()}")
print(f"✅ Inf                       : 0")

# 5) CSV'ye aktar
df_final.to_csv(OUTPUT_PATH, index=False)
print(f"\n💾 Dosya kaydedildi: {OUTPUT_PATH}")
df_final.head(10)

In [ ]:
# ============================================================
# HÜCRE 9: Doğrulama — Her Olayın Satır Sayısını Kontrol Et
# ============================================================

print("OLAY ÇALIŞMASI DOĞRULAMA RAPORU")
print("=" * 60)

validation = df_final.groupby("Olay_Ismi").agg(
    Satir=("T0_Goreceli_Gun", "count"),
    T_Min=("T0_Goreceli_Gun", "min"),
    T_Max=("T0_Goreceli_Gun", "max"),
    T0_Tarih=("T0_Tarih", "first")
).sort_values("T0_Tarih")

for _, row in validation.iterrows():
    status = "✅" if row["Satir"] >= 29 else "⚠️"
    print(f"{status} {row.name:<40} | {row['Satir']:>3} satır | "
          f"T=[{row['T_Min']:+d} → {row['T_Max']:+d}] | T0={row['T0_Tarih'].date()}")

print(f"\n{'='*60}")
print(f"Toplam satır : {len(df_final)}")
print(f"Toplam sütun : {len(df_final.columns)}")
print(f"Dosya boyutu : {OUTPUT_PATH}")